# Статический парсинг Яндекс.Карт

**Что собираю:**
- Название кафе
- Рейтинг
- Количество оценок
- Цены из меню

**Инструменты:** `requests`, `BeautifulSoup`, `re`, `pandas`, `matplotlib`

**Почему Яндекс.Карты?** Это основная платформа отзывов и информации о заведениях в России. Страницы кафе (название, рейтинг, меню) доступны в статическом HTML — подтверждено предварительным тестом.

In [ ]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd
import time
import os
import matplotlib.pyplot as plt

# Создаём папки, если их ещё нет
os.makedirs('../data/raw', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../figures', exist_ok=True)

print('Библиотеки загружены.')

## Шаг 1. Список кафе и URL

Выбираем 5 баблти-кафе в Москве. URL берём с их страниц на Яндекс.Картах.

In [ ]:
# Список кафе-конкурентов и их страницы на Яндекс.Картах
cafes = [
    {"cafe_name": "Nova",
     "url": "https://yandex.ru/maps/org/nova/232329106809/"},
    {"cafe_name": "Teapoke bubble tea",
     "url": "https://yandex.ru/maps/org/teapoke_bubble_tea/63621930136/"},
    {"cafe_name": "Jinju bubble tea",
     "url": "https://yandex.ru/maps/org/jinju_bubble_tea/232950636517/"},
    {"cafe_name": "Mani Bubble Tea",
     "url": "https://yandex.ru/maps/org/mani_bubble_tea/179312554027/"},
    {"cafe_name": "Teakwon-do Bubble Tea",
     "url": "https://yandex.ru/maps/org/teakwon_do_bubble_tea/63128021125/"},
]

# Заголовок запроса — имитируем обычный браузер
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

print(f'Кафе для парсинга: {len(cafes)}')

## Шаг 2. Функция парсинга одной страницы

Функция загружает страницу, извлекает текст и ищет нужные данные с помощью регулярных выражений (`re`).

In [ ]:
def parse_cafe_page(cafe_name, url):
    """
    Загружает страницу кафе на Яндекс.Картах.
    Возвращает словарь: название, рейтинг, кол-во оценок, цены из меню.
    """
    response = requests.get(url, headers=headers, timeout=10)
    soup = BeautifulSoup(response.text, 'lxml')

    # Получаем весь видимый текст страницы
    text = soup.get_text(separator=' ', strip=True)

    # Ищем рейтинг: паттерн вида "Рейтинг 4,7"
    rating_match = re.search(r'Рейтинг\s+(\d+[,\.]\d+)', text)
    rating = float(rating_match.group(1).replace(',', '.')) if rating_match else None

    # Ищем количество оценок: паттерн вида "520 оценок"
    reviews_match = re.search(r'(\d+)\s+оценок', text)
    num_reviews = int(reviews_match.group(1)) if reviews_match else None

    # Ищем все цены: числа перед символом ₽
    all_prices = re.findall(r'(\d{2,4})\s*₽', text)
    # Оставляем только реалистичные цены на напитки: от 100 до 1000 руб.
    prices = [int(p) for p in all_prices if 100 <= int(p) <= 1000]

    print(f'{cafe_name}: рейтинг={rating}, оценок={num_reviews}, цен найдено={len(prices)}')

    return {
        'cafe_name': cafe_name,
        'rating': rating,
        'num_reviews': num_reviews,
        'prices': prices,
        'source_url': url
    }

## Шаг 3. Запускаем парсинг

Делаем паузу 2 секунды между запросами, чтобы не перегружать сервер.

In [ ]:
results = []

for cafe in cafes:
    data = parse_cafe_page(cafe['cafe_name'], cafe['url'])
    results.append(data)
    time.sleep(2)  # пауза между запросами

print('\nПарсинг завершён')

## Шаг 4. Сохраняем данные о кафе

In [ ]:
# Таблица с общей информацией по каждому кафе
cafes_rows = []
for r in results:
    cafes_rows.append({
        'cafe_name':   r['cafe_name'],
        'rating':      r['rating'],
        'num_reviews': r['num_reviews'],
        'source_url':  r['source_url']
    })

df_cafes = pd.DataFrame(cafes_rows)
print(df_cafes)

df_cafes.to_csv('../data/raw/yandex_maps_cafes.csv', index=False, encoding='utf-8-sig')
print('\nСохранено: data/raw/yandex_maps_cafes.csv')

## Шаг 5. Сохраняем цены из меню

In [ ]:
# Таблица цен: одна строка = одна цена в меню одного кафе
prices_rows = []
for r in results:
    for price in r['prices']:
        prices_rows.append({
            'cafe_name':  r['cafe_name'],
            'price':      price,
            'source_url': r['source_url']
        })

df_prices = pd.DataFrame(prices_rows)
print(f'Всего записей с ценами: {len(df_prices)}')
print(df_prices.head(10))

df_prices.to_csv('../data/raw/yandex_maps_prices.csv', index=False, encoding='utf-8-sig')
print('\nСохранено: data/raw/yandex_maps_prices.csv')

---
# Анализ цен конкурентов

Загружаем сохранённые данные и смотрим на ценовой диапазон рынка баблти в Москве.

In [ ]:
df_prices = pd.read_csv('../data/raw/yandex_maps_prices.csv')

print('Статистика цен на напитки в баблти-кафе Москвы:')
print(f'  Минимальная цена:  {df_prices["price"].min()} ₽')
print(f'  Максимальная цена: {df_prices["price"].max()} ₽')
print(f'  Средняя цена:      {df_prices["price"].mean():.0f} ₽')
print(f'  Медианная цена:    {df_prices["price"].median():.0f} ₽')
print(f'  Всего позиций:     {len(df_prices)}')

## График 1: Распределение цен

In [ ]:
mean_price  = df_prices['price'].mean()
median_price = df_prices['price'].median()

plt.figure(figsize=(10, 5))
plt.hist(df_prices['price'], bins=15, color='steelblue', edgecolor='white')
plt.axvline(mean_price,   color='red',    linestyle='--', label=f'Среднее: {mean_price:.0f} ₽')
plt.axvline(median_price, color='orange', linestyle='--', label=f'Медиана: {median_price:.0f} ₽')
plt.title('Распределение цен на напитки в баблти-кафе Москвы')
plt.xlabel('Цена, ₽')
plt.ylabel('Количество позиций в меню')
plt.legend()
plt.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig('../figures/price_distribution.png', dpi=100)
plt.show()

![Частота тем во всех отзывах](../figures/price_distribution.png)

## График 2: Средняя цена по кафе

In [ ]:
avg_by_cafe = df_prices.groupby('cafe_name')['price'].mean().sort_values()

plt.figure(figsize=(10, 5))
plt.barh(avg_by_cafe.index, avg_by_cafe.values, color='steelblue')
plt.title('Средняя цена напитка по кафе-конкурентам')
plt.xlabel('Средняя цена, ₽')
plt.grid(axis='x', alpha=0.4)
plt.tight_layout()
plt.savefig('../figures/avg_price_by_cafe.png', dpi=100)
plt.show()

![Частота тем во всех отзывах](../figures/price_average.png)

## Выводы по ценам

На основе данных из меню конкурентов:

- Цены на баблти в Москве варьируются в диапазоне примерно **100-1000 ₽**
- Медианная цена позволяет ориентироваться на **средний ценовой сегмент** при разработке нашего меню
- Кафе с более высоким средним чеком, как правило, позиционируют себя как премиум (большой выбор топпингов, авторские напитки)
- Для нашего запуска рекомендуется ценовой диапазон, близкий к медиане рынка, чтобы быть конкурентоспособными

**Ограничение:** цены извлечены из текста страниц меню. Часть цифр может относиться к ценам на еду (салаты, закуски), а не только на напитки. Итоговые данные нужно интерпретировать как ценовой диапазон заведения, а не строго как цены только на баблти.